# 🚨 TriageAI — Local Deployment with Ollama + Gemma 4
## Offline Emergency Triage on Any Laptop · Ollama Special Prize Track

---

### What This Notebook Does
Deploys **TriageAI** as a named local Ollama model (`triageai`) backed by Gemma 4 E2B-IT.  
A custom **Modelfile** bakes the START triage protocol system prompt directly into the model so any bystander can run `ollama run triageai` and get expert first-aid guidance — **no internet, no cloud, no Python**.

### Why Offline Matters
> *In every major disaster — earthquakes, hurricanes, building collapses — cell towers and internet go down within the first hour.*  
> Emergency responders arrive minutes to hours later. TriageAI runs on a \$300 laptop with no connectivity.

### Architecture
```
Bystander Input (any language)
        │
        ▼
┌─────────────────────────────────────┐
│  Ollama Server  (127.0.0.1:11434)   │
│  ├─ Modelfile: START triage prompt  │
│  └─ Base: Gemma 4 E2B-IT (~2GB)     │
└──────────────┬──────────────────────┘
               │  /api/chat
               ▼
┌─────────────────────────────────────┐
│  Structured JSON Response           │
│  { triage_color, immediate_actions, │
│    do_not, dispatcher_script, ... } │
└─────────────────────────────────────┘
               │
               ▼
  Color-coded triage card (RED/YELLOW/GREEN/BLACK)
```

| Detail | Value |
|---|---|
| **Inference backend** | Ollama (local server, no cloud) |
| **Model** | Gemma 4 E2B-IT (fits on 4GB RAM laptops) |
| **Internet required** | ❌ No — fully offline after one-time pull |
| **Languages tested** | English · Spanish · Hindi |
| **Output format** | Structured JSON (7 fields, START-compliant) |
| **Prize target** | 🏆 Ollama Special Prize — $10,000 |

### Cloud vs. Local — Why Ollama?

| Aspect | Cloud API (OpenAI/Gemini) | Local Ollama + Gemma 4 |
|---|---|---|
| Internet required | ✅ Always | ❌ After initial pull |
| Works in disaster zones | ❌ No | ✅ Yes |
| Data privacy | ❌ Sent to servers | ✅ Never leaves device |
| Cost per query | ~\$0.01–0.05 | Free |
| Setup for volunteers | Complex API keys | `ollama run triageai` |
| Works on cheap laptop | ❌ No | ✅ 4GB RAM, CPU-only |

> 💡 **Fine-tuned models work here too.** The Unsloth fine-tuned adapter (see `02_unsloth_finetune.ipynb`) can be loaded in place of the base model — giving triage-specific responses trained on real emergency protocols, with zero change to the deployment pipeline.


## Step 0: Setup — Install Dependencies & Imports

> **Kaggle settings required:** Enable **Internet ON** in the notebook settings before running.


In [ ]:
%%capture
!pip install -q requests


In [ ]:
# All imports consolidated here
import subprocess
import os
import shutil
import time
import socket
import json
import requests
from IPython.display import display, HTML

print("Imports OK")


## Step 1: Install Ollama

Ollama is a lightweight local inference server that manages model downloads, quantization, and GPU/CPU routing automatically. We install it via the official install script and verify the binary is accessible.


In [ ]:
# 1. Verify internet connectivity (required for pulling model)
def check_internet():
    try:
        socket.setdefaulttimeout(5)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
        return True
    except Exception:
        return False

if not check_internet():
    raise RuntimeError(
        'No internet detected. Enable Internet in Kaggle Settings, then Save & Run All.'
    )
print('✓ Internet: OK')

# 2. Install zstd (dependency of Ollama installer on Ubuntu)
subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True, capture_output=True)
print('✓ zstd: OK')

# 3. Install Ollama via official script
print('Installing Ollama...')
install_result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
if install_result.returncode != 0:
    print('STDOUT:', install_result.stdout[-500:])
    print('STDERR:', install_result.stderr[-500:])
    raise RuntimeError('Ollama install failed.')
print('✓ Ollama installed.')

# 4. Locate binary
OLLAMA_BIN = None
for _p in [
    shutil.which('ollama'),
    '/usr/local/bin/ollama',
    '/usr/bin/ollama',
    os.path.expanduser('~/.local/bin/ollama'),
]:
    if _p and os.path.isfile(_p):
        OLLAMA_BIN = _p
        break

if not OLLAMA_BIN:
    raise FileNotFoundError('Ollama binary not found after install.')

version = subprocess.check_output([OLLAMA_BIN, '--version'], text=True).strip()
print(f'✓ Binary : {OLLAMA_BIN}')
print(f'✓ Version: {version}')


## Step 2: Start Ollama Server & Pull Gemma 4 E2B

Ollama runs as a local HTTP server on port 11434. We start it as a background process, wait for it to be ready, then pull `gemma4:e2b` — the 2B-parameter instruction-tuned variant that fits comfortably on a 4GB RAM laptop.


In [ ]:
# Start Ollama server in background
env = os.environ.copy()
env['OLLAMA_HOST'] = '127.0.0.1:11434'
proc = subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env
)

# Wait for server (up to 30s)
print('Starting Ollama server...', end='', flush=True)
for i in range(30):
    try:
        r = requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f' ready in {i+1}s ✓')
            break
    except Exception:
        pass
    time.sleep(1)
    print('.', end='', flush=True)
else:
    print(' timeout — server may still be starting, continuing anyway')

print(f'Ollama server PID: {proc.pid}')

# Pull Gemma 4 E2B (instruction-tuned variant, ~2GB)
print('\nPulling gemma4:e2b (this takes a few minutes on first run)...')
result = subprocess.run([OLLAMA_BIN, 'pull', 'gemma4:e2b'], capture_output=True, text=True)
if result.returncode != 0:
    print("  gemma4:e2b unavailable, trying gemma4:e4b...")
    result = subprocess.run([OLLAMA_BIN, 'pull', 'gemma4:e4b'], capture_output=True, text=True)
    BASE_MODEL = 'gemma4:e4b'
else:
    BASE_MODEL = 'gemma4:e2b'

if result.returncode == 0:
    print(f'✓ Model pulled: {BASE_MODEL}')
    print('  Ollama automatically serves the instruction-tuned variant for Gemma 4.')
else:
    print(f'✗ Pull failed: {result.stderr[:300]}')
    BASE_MODEL = 'gemma4:e2b'

print(f'\nBase model set to: {BASE_MODEL}')


## Step 3: Create the TriageAI Modelfile

An Ollama **Modelfile** is like a Dockerfile for AI models. It specifies:
- `FROM` — which base model to use
- `SYSTEM` — the system prompt baked permanently into the model
- `PARAMETER` — inference defaults (temperature, top_p, output length)

Once created, running `ollama run triageai` gives a fully configured emergency triage assistant — no Python, no API keys, no internet required.

**The system prompt implements the START triage protocol:**  
- **RED** = Immediate (life-threatening)  
- **YELLOW** = Delayed (serious but stable)  
- **GREEN** = Minor (walking wounded)  
- **BLACK** = Expectant (unsurvivable with current resources)


In [ ]:
SYSTEM_PROMPT = '''You are TriageAI, an emergency bystander first-aid assistant.
You follow the START (Simple Triage and Rapid Treatment) triage protocol.

For every emergency described, output ONLY a single valid JSON object with these fields:
- emergency_type: string describing the emergency category
- triage_color: RED, YELLOW, GREEN, or BLACK
- triage_label: IMMEDIATE, DELAYED, MINOR, or EXPECTANT
- life_threats: array of life-threatening conditions identified
- immediate_actions: array of step-by-step action strings for the bystander
- do_not: array of things the bystander must NOT do
- dispatcher_script: short script to read directly to the 911 dispatcher

START triage criteria:
- RED (IMMEDIATE): not breathing / breathing >30/min / no radial pulse / altered mental status
- YELLOW (DELAYED): breathing, has pulse, follows commands but cannot walk
- GREEN (MINOR): walking wounded, minor injuries
- BLACK (EXPECTANT): no breathing after airway repositioning, unsurvivable injuries

Output only valid JSON. No markdown fences. No explanation outside the JSON.'''

MODELFILE = f'''FROM {BASE_MODEL}

SYSTEM """{SYSTEM_PROMPT}"""

PARAMETER temperature 0.7
PARAMETER top_p 0.95
PARAMETER num_predict 800
'''

modelfile_path = "/tmp/TriageAI.Modelfile"
with open(modelfile_path, "w") as f:
    f.write(MODELFILE)

print("=" * 60)
print("MODELFILE CONTENTS:")
print("=" * 60)
# Print header lines only (system prompt is long)
for line in MODELFILE.split('\n')[:6]:
    print(line)
print(f"  ... (system prompt: {len(SYSTEM_PROMPT)} chars, START protocol) ...")
print("PARAMETER temperature 0.7")
print("PARAMETER top_p 0.95")
print("PARAMETER num_predict 800")
print()

# Create the model in Ollama
result = subprocess.run(
    [OLLAMA_BIN, 'create', 'triageai', '-f', modelfile_path],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓ TriageAI model created in Ollama registry.')
    print('  Deployment command: ollama run triageai')
else:
    print(f'✗ Model creation failed: {result.stderr[:300]}')

# Verify model is registered
print()
print("=" * 60)
print("REGISTERED MODELS (ollama list):")
print("=" * 60)
verify = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
print(verify.stdout)
assert 'triageai' in verify.stdout, 'ERROR: triageai not found in Ollama registry!'
print('✓ triageai confirmed in registry.')


## Step 4: Triage Helper Functions

Two functions power the demo:
- `triage_ollama()` — sends a scenario to the local Ollama API and parses the JSON response
- `render_card()` — renders a color-coded HTML triage card matching the START protocol colors

The 300s timeout handles Kaggle's shared CPU (first inference is slow due to model loading; subsequent calls are fast).


In [ ]:
COLORS = {
    'RED':    ('#d32f2f', '#ffffff', 'IMMEDIATE'),
    'YELLOW': ('#f9a825', '#000000', 'DELAYED'),
    'GREEN':  ('#2e7d32', '#ffffff', 'MINOR'),
    'BLACK':  ('#212121', '#ffffff', 'EXPECTANT'),
}

def triage_ollama(scenario, model='triageai'):
    """Send scenario to local Ollama API, return parsed triage JSON."""
    start = time.time()
    try:
        r = requests.post(
            'http://127.0.0.1:11434/api/chat',
            json={
                'model': model,
                'messages': [{'role': 'user', 'content': scenario}],
                'stream': False,
                'options': {'num_predict': 800, 'temperature': 0.7, 'top_p': 0.95},
            },
            timeout=300,  # First inference loads model into memory; allow up to 5 min
        )
        raw = r.json().get('message', {}).get('content', '')
    except Exception as e:
        return {
            'emergency_type': 'connection_error', 'triage_color': 'YELLOW',
            'triage_label': 'DELAYED', 'life_threats': [],
            'immediate_actions': [f'Error contacting Ollama: {e}'],
            'do_not': [], 'dispatcher_script': 'Call 911',
            '_elapsed': time.time() - start, '_raw': ''
        }
    elapsed = time.time() - start

    # Extract JSON from response (model may prepend/append text)
    try:
        start_idx = raw.index('{')
        end_idx   = raw.rindex('}') + 1
        result = json.loads(raw[start_idx:end_idx])
    except Exception:
        result = {
            'emergency_type': 'parse_error', 'triage_color': 'YELLOW',
            'triage_label': 'DELAYED', 'life_threats': [],
            'immediate_actions': [raw[:400] if raw else 'Empty response'],
            'do_not': [], 'dispatcher_script': 'Call 911'
        }
    result['_elapsed'] = elapsed
    result['_raw'] = raw
    return result


def render_card(result, title):
    """Render a color-coded START triage card as HTML."""
    color_hex, text_color, label = COLORS.get(
        result.get('triage_color', 'YELLOW'), ('#f9a825', '#000000', 'DELAYED')
    )
    triage_color = result.get('triage_color', 'YELLOW')
    elapsed      = result.get('_elapsed', 0)
    actions_html = ''.join(f'<li style="margin:4px 0">{a}</li>' for a in result.get('immediate_actions', []))
    donot_html   = ''.join(f'<li style="margin:4px 0;color:#b71c1c">{d}</li>' for d in result.get('do_not', []))
    threats      = ', '.join(result.get('life_threats', [])) or 'None identified'

    html = f"""
    <div style='border:3px solid {color_hex};border-radius:10px;padding:16px;margin:12px 0;
                font-family:system-ui,sans-serif;background:#ffffff;color:#111111;'>
      <div style='background:{color_hex};color:{text_color};padding:12px 16px;
                  border-radius:6px;margin-bottom:14px;display:flex;justify-content:space-between;align-items:center;'>
        <strong style='font-size:1.3em;color:{text_color};'>{triage_color} — {label}</strong>
        <span style='font-size:0.85em;color:{text_color};opacity:0.9;'>Ollama · Gemma 4 E2B · {elapsed:.1f}s</span>
      </div>
      <p style='color:#111111;margin:6px 0;'><strong style='color:#111111;'>Scenario:</strong>
        <span style='color:#333;'>{title}</span></p>
      <p style='color:#111111;margin:6px 0;'><strong style='color:#111111;'>Emergency type:</strong>
        <span style='color:#333;'>{result.get('emergency_type', 'unknown')}</span></p>
      <p style='color:#111111;margin:6px 0;'><strong style='color:#c62828;'>⚠ Life threats:</strong>
        <span style='color:#333;'>{threats}</span></p>
      <div style='background:#fff8e1;border-left:4px solid #f9a825;padding:10px 14px;
                  border-radius:4px;margin:10px 0;'>
        <strong style='color:#e65100;'>⚡ Immediate Actions:</strong>
        <ol style='margin:6px 0;padding-left:20px;color:#111111;'>{actions_html}</ol>
      </div>
      <div style='background:#ffebee;border-left:4px solid #d32f2f;padding:10px 14px;
                  border-radius:4px;margin:10px 0;'>
        <strong style='color:#b71c1c;'>🚫 DO NOT:</strong>
        <ul style='margin:6px 0;padding-left:20px;color:#111111;'>{donot_html}</ul>
      </div>
      <div style='background:#e3f2fd;border-left:4px solid #1565c0;padding:10px 14px;
                  border-radius:4px;margin:10px 0;font-size:0.95em;'>
        <strong style='color:#0d47a1;'>📞 Say to dispatcher:</strong>
        <span style='color:#111111;'> {result.get('dispatcher_script', '')}</span>
      </div>
    </div>"""
    display(HTML(html))
    return result


print('✓ Helper functions defined.')


## Step 5: Live Triage Tests — 3 Languages, 3 Emergencies

Each test sends a real-world emergency scenario to the local Ollama server.  
The model responds with a structured JSON triage assessment rendered as a color-coded card.

**No internet is used for inference** — all calls go to `127.0.0.1:11434` (localhost).


In [ ]:
print("=" * 60)
print("TEST 1: Severe Arm Laceration — English")
print("=" * 60)
r1 = triage_ollama(
    "My friend fell on broken glass and has a deep cut on his forearm. "
    "There is a lot of blood spurting out and he is getting pale. "
    "We are at a construction site. What do I do?"
)
render_card(r1, "Severe Arm Laceration — English")


In [ ]:
print("=" * 60)
print("TEST 2: Earthquake Victim Trapped — Spanish")
print("=" * 60)
r2 = triage_ollama(
    "Hubo un terremoto fuerte. Mi vecina esta atrapada bajo escombros. "
    "Puedo ver su brazo pero no responde. "
    "Hay cables electricos caidos cerca. Que hago?"
)
render_card(r2, "Earthquake Victim Trapped — Spanish")


In [ ]:
print("=" * 60)
print("TEST 3: Cardiac Arrest — Hindi")
print("=" * 60)
r3 = triage_ollama(
    "\u092e\u0947\u0930\u0947 \u092a\u093f\u0924\u093e\u091c\u0940 \u0905\u091a\u093e\u0928\u0915 "
    "\u0938\u0940\u0928\u0947 \u092e\u0947\u0902 \u0926\u0930\u094d\u0926 \u0915\u0940 "
    "\u0936\u093f\u0915\u093e\u092f\u0924 \u0915\u0930\u0924\u0947 \u0939\u0941\u090f "
    "\u0917\u093f\u0930 \u0917\u090f \u0939\u0948\u0902\u0964 "
    "\u0935\u0947 \u0938\u093e\u0902\u0938 \u0928\u0939\u0940\u0902 \u0932\u0947 \u0930\u0939\u0947 \u0939\u0948\u0902\u0964 "
    "\u0915\u0943\u092a\u092f\u093e \u092e\u0926\u0926 \u0915\u0930\u0947\u0902!"
)
render_card(r3, "Cardiac Arrest — Hindi")


## Step 6: Performance Benchmark — 3 Additional Scenarios

Testing three more scenarios to measure latency and validate correct triage classification across different emergency types.


In [ ]:
benchmark_cases = [
    ("Minor burn on hand",   "GREEN",  "I touched a hot pan and have a small red burn on my hand. It hurts but I can move my fingers."),
    ("Choking adult",        "RED",    "A man at a restaurant is choking on food. He cannot breathe or speak and is turning blue."),
    ("Multi-vehicle crash",  "RED",    "Three-car accident on highway. One person unconscious, not moving. I can smell gasoline."),
]

bench_results = []
print(f"{'Case':<25} {'Expected':>8} {'Got':>8} {'Match':>6} {'Latency':>9} {'Actions':>8}")
print("-" * 70)
for name, expected, scenario in benchmark_cases:
    rb = triage_ollama(scenario)
    got     = rb.get('triage_color', '?')
    match   = '✓' if got == expected else '✗'
    elapsed = rb.get('_elapsed', 0)
    n_steps = len(rb.get('immediate_actions', []))
    print(f"  {name:<23} {expected:>8} {got:>8} {match:>6} {elapsed:>7.1f}s {n_steps:>8} steps")
    bench_results.append({'name': name, 'expected': expected, 'got': got,
                          'elapsed': elapsed, 'steps': n_steps, 'result': rb})

print()
print("Note: Latency is on shared Kaggle CPU. On a local laptop with GPU, typically 2-5x faster.")
print("All inference is local — no data sent to any cloud service.")


## Step 7: Results Summary


In [ ]:
all_results = [
    ('Severe Arm Laceration', 'English', 'RED',    r1),
    ('Earthquake Trapped',    'Spanish', 'RED',    r2),
    ('Cardiac Arrest',        'Hindi',   'RED',    r3),
] + [
    (b['name'], 'English', b['expected'], b['result'])
    for b in bench_results
]

rows = []
for name, lang, expected, res in all_results:
    got     = res.get('triage_color', '?')
    elapsed = res.get('_elapsed', 0)
    steps   = len(res.get('immediate_actions', []))
    status  = '✅ Pass' if got == expected else f'⚠️ Got {got}'
    rows.append((name, lang, expected, got, status, f'{elapsed:.1f}s', steps))

header = f"{'Test Case':<28} {'Lang':<10} {'Expected':<10} {'Got':<10} {'Status':<12} {'Latency':<10} {'Steps'}"
print(header)
print('-' * 90)
for row in rows:
    print(f"  {row[0]:<26} {row[1]:<10} {row[2]:<10} {row[3]:<10} {row[4]:<12} {row[5]:<10} {row[6]}")

passed = sum(1 for r in rows if 'Pass' in r[4])
total  = len(rows)
print()
print(f"Result: {passed}/{total} correct triage classifications")
print(f"Multilingual coverage: English, Spanish, Hindi — same structured output format")


## 🏆 Ollama Prize Checklist

| Requirement | Status | Evidence |
|---|---|---|
| Ollama as inference backend | ✅ | `ollama serve` + `/api/chat` endpoint, no cloud calls |
| Custom Modelfile | ✅ | `FROM gemma4:e2b` + embedded START triage system prompt |
| Named deployable model | ✅ | `ollama create triageai` — confirmed in `ollama list` |
| Multilingual support | ✅ | English, Spanish, Hindi tested — same JSON schema |
| Structured output | ✅ | 7-field JSON per response (`triage_color`, `immediate_actions`, `do_not`, `dispatcher_script`, ...) |
| Offline capability | ✅ | All inference to `127.0.0.1:11434` — no internet after model pull |
| Works on consumer hardware | ✅ | Gemma 4 E2B: ~2GB VRAM, runs on CPU-only 4GB RAM laptops |
| One-command deployment | ✅ | `ollama run triageai` — no Python, no API keys |

---

## Summary

TriageAI is deployed as a named Ollama model (`triageai`) backed by Gemma 4 E2B-IT. The custom Modelfile permanently embeds the START triage protocol system prompt, so any bystander can run `ollama run triageai` on their laptop and receive structured, actionable emergency guidance in multiple languages — with zero internet, zero cloud, and zero setup beyond the initial model pull.

**Key outcomes:**
- **6 emergency scenarios** tested across 3 languages — all produce structured `triage_color` + `immediate_actions` + `do_not` + `dispatcher_script`
- **One-command deployment**: `ollama run triageai` works immediately after `ollama create`
- **Privacy-first**: patient descriptions never leave the device
- **Disaster-ready**: runs offline on inexpensive hardware (4GB RAM, no GPU required)

### System Requirements for Local Deployment

| Resource | Minimum | Recommended |
|---|---|---|
| RAM | 4 GB | 8 GB |
| Disk | 3 GB | 5 GB |
| GPU | Not required | Any CUDA/Metal GPU |
| OS | macOS / Linux / Windows WSL2 | Any |
| Download (one-time) | ~2 GB | — |

### Quick Start (for anyone to reproduce)
```bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Pull base model (one-time, ~2GB)
ollama pull gemma4:e2b

# Create TriageAI model with system prompt
ollama create triageai -f TriageAI.Modelfile

# Run — works fully offline from here
ollama run triageai
```

---
*TriageAI · Ollama $10K Special Prize · Gemma 4 Good Hackathon 2026*  
*Project: https://github.com/Kalyankr/triageai · Demo: https://huggingface.co/spaces/kalyanreddy77/triageai*
